In [0]:
schema = 'fifa_bi_dev.gold_schema'

team_all_time_standing

In [0]:
from pyspark.sql.functions import *

matches = spark.read.table('fifa_bi_dev.silver_schema.silver_world_cup_matches')

team_1_standing = matches.select(
    'tournament_name', 
    'year',
    'match_id', 
    col('team_1_code').alias('team_code'), 
    col('team_1_country').alias('team_country'), 
    col('team_1_goals').alias('goals_for'),
    col('team_2_goals').alias('goals_against'),
    col('team_2_code').alias('opponent_team_code'),
    col('team_2_country').alias('opponent_country'),
    col('round'),
    col('match_stage'),
    when(col('winner_country') == lit('--'), lit('Draw'))
    .when(col('winner_country') == col('team_1_country'), lit('Win'))
    .otherwise(lit('Lost')).alias('result')
    )  

team_2_standing = matches.select(
    'tournament_name',
    'year', 
    'match_id', 
    col('team_2_code').alias('team_code'), 
    col('team_2_country').alias('team'), 
    col('team_2_goals').alias('goals_for'),
    col('team_1_goals').alias('goals_against'),
    col('team_1_code').alias('opponent_team_code'),
    col('team_1_country').alias('opponent'),
    col('round'),
    col('match_stage'),
    when(col('winner_country') == lit('--'), lit('Draw'))
    .when(col('winner_country') == col('team_2_country'), lit('Win'))
    .otherwise(lit('Lost')).alias('result')
    )  

team_all_time_standing = team_1_standing.union(team_2_standing)


team_all_time_standing.show()


In [0]:
team_all_time_standing.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f'{schema}.gold_team_all_time_standing')
print(f'table_name: gold_team_all_time_standing is updated')